# Suite PYS — Python3 simplified by Python's rules

`Python3.Simplifiers.OfAny(term, scope)` simplifies a Python3 term by Python's rules, CPython 3.14's (mbse-basis
[CAP-295: simplification of single-dialect expressions with literal leaves](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-295) to
[CAP-297: simplification in every binding state](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-297)): Python runs
them, and TypeScript models them over JavaScript's values, the same tables of values and errors in both. What the scope
decides folds into the literal of its value (a basic scalar's, or a constant's: `None`, `...`, a complex number), a
list, a tuple, a dict or a non-empty set of such values into its display, a closed constructor expression; a value with
none (an empty set, a slice, a closure, a template, a generator) keeps its subterm, its own subterms simplified; and what Python leaves to a statement, to its
implementation or to its platform stays, named with its class in the evaluation inventory (`residues`): `yield` and
`await` (`d`), `1 is 1` (`b`), a complex power or a locale's format (`a`). What Python makes an error still raises,
by the same class and message. Terms are built through the dialect's writers, in a transaction (`term`); `text` gives
a residual's source, as the printer writes it.

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.Python.Python3 import Simplifiers as PV, Expressions as P, Text as PText
from mbse.Expressions.Framework import Terms as F
from mbse.Expressions.Framework.Simplifiers import NotEvaluated
from mbse.Schemas.Framework import Builders
from support import raises

store = Builders.Store()  # the framework, the standard dialects Python3's complex numbers use, and the Python family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
                PythonTypes.install, PythonPatterns.install]:
    store.transaction(install)


def term(spec):
    """The Python3 term `spec` writes, a native value as its literal, the root of a holder built in a transaction."""
    with store.transaction():
        return F.term(store.Expressions.Framework.Expression(lambda e: e.root(P.DIALECT.spec(spec))))


def source(spec):
    """The source of the term `spec` writes."""
    return PText.ToText(term(spec))


def simplified(spec, variables=None):
    """The simplification of the term `spec` writes, in a scope binding `variables`."""
    return PV.OfAny(term(spec), PV.Scope(variables or {}))


def value(spec, variables=None):
    """The value the term `spec` writes simplifies to: its literal root's."""
    return simplified(spec, variables).evaluate()


def text(spec, variables=None):
    """The source of the residual of the term `spec` writes, simplified in a scope binding `variables`."""
    return PText.ToText(term(simplified(spec, variables).residual))


def reasons(spec, variables=None):
    """Why what stayed in the residual stayed: each residue's reason and class."""
    return [(residue.reason, residue.category) for residue in simplified(spec, variables).residues]


def folds(rows):
    """Each (expression, variables, value) folds to a literal root of that value, of its type."""
    for spec, variables, expected in rows:
        found = value(spec, variables)
        assert found == expected and type(found) is type(expected), (source(spec), found, expected)


def stays(rows):
    """Each (expression, variables, residual) simplifies to that residual, its root no literal and nothing named: a
    value with no literal, a display, or a term whose subterms simplified."""
    for spec, variables, expected in rows:
        result = simplified(spec, variables)
        shown = PText.ToText(term(result.residual))
        assert not result.literal and not result.residues and shown == expected, (source(spec), shown, expected)


def named(rows):
    """Each (expression, variables, reason, class) leaves one subterm in the residual, named by that reason and class."""
    for spec, variables, reason, category in rows:
        assert reasons(spec, variables) == [(reason, category)], (source(spec), reasons(spec, variables))


def fails(rows):
    """Each (expression, variables, error, message) raises as Python raises it."""
    for spec, variables, error, message in rows:
        with raises(error, match=message):
            simplified(spec, variables)


# the builders of the terms the cases write
n, kw, st, fv, comp = P.name, P.keyword, P.starred, P.formattedvalue, P.comprehension
x, y, z, v, m = n("x"), n("y"), n("z"), n("v"), n("m")
inf, nan = math.inf, math.nan
s = lambda spec: P.call("str", spec)  # noqa: E731: `str(spec)`, which shows what equality cannot (a NaN, a sign)
c1, c2, one = P.constant(1j), P.binop("+", 1, P.constant(2j)), P.constant(1 + 1j)  # 1j, 1 + 2j, (1+1j)

<a id="PYS-01"></a>
## PYS-01 · Constants: `None`, `...` and complex numbers fold, and a value is written back as its literal, or as its display

In [ ]:
# `None`, `...` and a complex number are Python3's constants: each folds to its value
folds([
    (P.constant(None), {}, None), (P.unaryop("not", P.constant(None)), {}, True), (s(P.constant(...)), {}, "Ellipsis"),
    (P.constant(2j), {}, 2j), (lambda u: getattr(u, P.CONSTANT)(lambda k: k.complex(lambda c: c.imag(2.0))), {}, 2j),
    (s(P.constant(1.5j)), {}, "1.5j"), (s(P.constant(2j)), {}, "2j"),
])
assert value(P.constant(...)) is Ellipsis and simplified(P.constant(None)).literal
fails([
    (lambda u: getattr(u, P.CONSTANT)(lambda r: r), {}, ValueError, "a constant needs a singleton or a complex number"),
    (P.binop("+", P.constant(...), 1), {}, TypeError, "unsupported operand type(s) for +: 'ellipsis' and 'int'"),
])
# A value is written back as its literal, a constant's among them: a literal root
for bound, written in [(None, "None"), (..., "..."), (1 + 2j, "(1+2j)"), (-2j, "(-0-2j)"), (-1.5, "-1.5"), (-3, "-3"),
                       (True, "True"), ("a", "'a'"), (b"z", "b'z'"), (inf, "float('inf')"), (nan, "float('nan')")]:
    assert text(x, {"x": bound}) == written and simplified(x, {"x": bound}).literal, written
# A list, a tuple, a dict or a set of values that have literals is written back as its display, a term, not a
# literal; a value with no literal keeps its term
stays([
    (P.tuple_(1, P.binop("+", 1, 1)), {}, "(1, 2)"), (P.tuple_(1), {}, "(1,)"), (x, {"x": (1, None, ...)}, "(1, None, ...)"),
    (x, {"x": {(1, 2): [3, (4,)]}}, "{(1, 2): [3, (4,)]}"), (x, {"x": []}, "[]"), (x, {"x": {}}, "{}"),
    (x, {"x": {2, 1}}, "{1, 2}"), (x, {"x": set()}, "x"),  # a non-empty set's display, its items in order (PYS-13)
])
with raises(NotEvaluated, match="the root is not a closed constructor expression: its value has no closed constructor expression"):
    simplified(P.lambda_(1)).evaluate()

<a id="PYS-02"></a>
## PYS-02 · Complex arithmetic as CPython 3.14 computes it: a real operand not made complex, infinities recovered, integer powers, `abs` correctly rounded; a power that needs the platform's functions stays, named

In [ ]:
folds([
    (c2, {}, 1 + 2j), (P.binop("*", c1, c1), {}, -1 + 0j), (P.binop("-", 1.5, c2), {}, 0.5 - 2j),
    (P.binop("/", c2, P.binop("-", 3, P.constant(4j))), {}, -0.2 + 0.4j), (P.binop("/", 1, c2), {}, 0.2 - 0.4j),
    (P.binop("**", P.binop("+", 1, c1), -2), {}, -0.5j), (P.binop("**", P.binop("+", 1, c1), 3), {}, -2 + 2j),
    (P.unaryop("not", P.constant(0j)), {}, True), (P.attribute(c2, "imag"), {}, 2.0),
    (P.compare("==", c1, P.binop("*", c1, 1)), {}, True), (P.compare("==", P.binop("+", 2, P.constant(0j)), 2), {}, True),
    (P.compare("==", 2, P.binop("+", 2, P.constant(0j))), {}, True), (s(P.binop("-", 1, P.constant(2j))), {}, "(1-2j)"),
    (s(P.binop("+", P.constant(complex(1, -0.0)), 1.0)), {}, "(2-0j)"),  # a real is not made complex (3.14)
    (s(P.binop("+", c1, 1.5)), {}, "(1.5+1j)"), (s(P.binop("-", c1, 1.5)), {}, "(-1.5+1j)"), (s(P.binop("*", 2.0, c1)), {}, "2j"),
    (s(P.binop("/", c1, 2.0)), {}, "0.5j"), (s(P.unaryop("-", c2)), {}, "(-1-2j)"), (s(P.unaryop("+", c1)), {}, "1j"),
    (s(P.binop("+", c1, c1)), {}, "2j"), (s(P.binop("-", c1, c1)), {}, "0j"), (s(P.binop("/", 1, P.constant(2 + 1j))), {}, "(0.4-0.2j)"),
    (s(P.binop("/", one, P.constant(4 + 1j))), {}, "(0.29411764705882354+0.17647058823529413j)"),  # Smith's algorithm
    (s(P.binop("**", P.constant(0j), 0.5)), {}, "0j"),
])
# Infinities and NaNs in products and quotients, recovered as C11's Annex G does
folds([
    (s(P.binop("*", x, y)), {"x": complex(inf, inf), "y": 1 + 0j}, "(inf+infj)"), (s(P.binop("*", y, x)), {"x": complex(inf, inf), "y": 1 + 0j}, "(inf+infj)"),
    (s(P.binop("*", x, y)), {"x": complex(nan, 1e300), "y": 1e300 + 1e300j}, "(-inf+infj)"), (s(P.binop("*", x, one)), {"x": complex(nan, nan)}, "(nan+nanj)"),
    (s(P.binop("/", one, x)), {"x": complex(nan, 1)}, "(nan+nanj)"), (s(P.binop("/", x, one)), {"x": complex(inf, nan)}, "(inf-infj)"),
    (s(P.binop("/", one, x)), {"x": complex(inf, inf)}, "0j"), (s(P.binop("/", 1, x)), {"x": complex(nan, 1)}, "(nan+nanj)"),
    (s(P.binop("/", x, one)), {"x": complex(-inf, nan)}, "(-inf+infj)"), (s(P.binop("/", x, one)), {"x": complex(nan, inf)}, "(inf+infj)"),
    (s(P.binop("/", one, x)), {"x": complex(inf, nan)}, "0j"), (s(P.binop("/", one, x)), {"x": complex(nan, inf)}, "-0j"),
])
# `abs` of a complex number, correctly rounded in both languages
folds([
    (P.call("abs", P.binop("+", 3, P.constant(4j))), {}, 5.0), (P.call("abs", P.constant(3j)), {}, 3.0),
    (P.call("abs", P.binop("+", x, c1)), {"x": inf}, inf), (s(P.call("abs", P.binop("+", x, c1))), {"x": nan}, "nan"),
    (P.call("abs", x), {"x": complex(5e-324, 5e-324)}, 5e-324),
])
# A complex power whose exponent is no integer, and a real power whose value is complex, need the platform's `pow`,
# `atan2`, `cos` and `sin`: each stays, named
named([
    (P.binop("**", c1, 0.5), {}, "Python3 binop '**' of a complex power is not simplified", "a"),
    (P.binop("**", -8, 0.5), {}, "Python3 binop '**' of a complex power is not simplified", "a"),
])
fails([
    (P.binop("/", 1, P.constant(0j)), {}, ZeroDivisionError, "division by zero"),
    (P.binop("/", c1, 0.0), {}, ZeroDivisionError, "division by zero"), (P.binop("/", c1, P.constant(0j)), {}, ZeroDivisionError, "division by zero"),
    (P.binop("**", P.constant(0j), -1), {}, ZeroDivisionError, "zero to a negative or complex power"),
    (P.binop("**", P.constant(0j), c1), {}, ZeroDivisionError, "zero to a negative or complex power"),
    (P.binop("**", P.constant(1e200 + 1e200j), 2), {}, OverflowError, "complex exponentiation"),
    (P.call("abs", P.binop("+", 1.5e308, P.constant(1.5e308j))), {}, OverflowError, "absolute value too large"),
    (P.compare("<", c1, c1), {}, TypeError, "'<' not supported between instances of 'complex' and 'complex'"),
    (P.binop("//", c1, 2), {}, TypeError, "unsupported operand type(s) for //: 'complex' and 'int'"),
])

<a id="PYS-03"></a>
## PYS-03 · Parentheses, adjacent strings and named expressions

In [ ]:
folds([
    (P.binop("*", P.parenthesized(P.binop("+", 1, 2)), 3), {}, 9), (P.concatenatedstring("a", "b"), {}, "ab"),
    (P.concatenatedstring(b"a", b"b"), {}, b"ab"), (P.binop("+", P.namedexpr("y", 2), 1), {}, 3),
    (P.namedexpr("y", x), {"x": 1.5}, 1.5),
])
stays([(P.binop("*", P.parenthesized(P.binop("+", x, P.binop("+", 1, 2))), 3), {}, "(x + 3) * 3")])
fails([
    (P.concatenatedstring(b"a", "b"), {}, TypeError, "cannot mix bytes and nonbytes literals"),
    (P.binop("+", b"a", "b"), {}, TypeError, "can't concat str to bytes"),
])

<a id="PYS-04"></a>
## PYS-04 · f-strings: conversions, debug fields, nested format specs and escapes; a residual's fields respelled; a field whose text is an implementation's, and Unicode's names, stay, named

In [ ]:
folds([
    (P.joinedstr(["x=", fv(x), "!"]), {"x": 1.5}, "x=1.5!"), (P.joinedstr([fv(x, "r", (">8",))]), {"x": "a"}, "     'a'"),
    (P.joinedstr([fv(x, None, None, True, "x=")]), {"x": (1, None)}, "x=(1, None)"),
    (P.joinedstr([fv(x, "a")]), {"x": "é"}, "'\\xe9'"), (P.joinedstr([fv(x, "a")]), {"x": "\u2028\U0001F600"}, "'\\u2028\\U0001f600'"),
    (P.joinedstr([fv(x, "a")]), {"x": "α"}, "'\\u03b1'"),
    (P.joinedstr([fv(x, "s")]), {"x": [1.0, None]}, "[1.0, None]"),
    (P.joinedstr([fv(x, None, (fv(n("w")), ".", fv(n("p")), "f"))]), {"x": 3.14159, "w": 8, "p": 3}, "   3.142"),
    (P.joinedstr(["{{", fv(1), "}}\\n\\x41\\u00e9\\101\\q"]), {}, "{1}\nAéA\\q"), (P.joinedstr(["\\n"], "rf"), {}, "\\n"),
    (P.joinedstr([fv(x, "r")]), {"x": {"a": (1,)}}, "{'a': (1,)}"), (P.joinedstr([fv(P.slice_(1, 2), "r")]), {}, "slice(1, 2, None)"),
    (P.joinedstr([]), {}, ""), (P.joinedstr(["a", fv(1)], None), {}, "a1"), (P.joinedstr([fv(x, None, ())]), {"x": 1}, "1"),
    (P.joinedstr([fv(x, None, None, True)]), {"x": 1}, "1"),
])
# A field whose term simplified loses its text as written: a debug field's label is written before it, with `!r` unless
# it has a conversion or a format spec
stays([
    (P.joinedstr([fv(P.binop("+", x, P.binop("+", 1, 2)), None, None, True, "x + (1+2)=")]), {}, "f'x + (1+2)={x + 3!r}'"),
    (P.joinedstr([fv(P.binop("+", x, P.binop("+", 1, 2)), None, (">5",), True, "x=")]), {}, "f'x={x + 3:>5}'"),
    (P.joinedstr([fv(P.binop("+", x, P.binop("+", 1, 2)), None, (), False, "x + (1+2)")]), {}, "f'{x + 3:}'"),
    (P.joinedstr([fv(P.binop("+", x, P.binop("+", 1, 2)), None, None, False, "x + (1+2)")]), {}, "f'{x + 3}'"),
    (P.let_("y", 2, P.joinedstr([fv(x), fv(y)])), {}, "f'{x}{2}'"),
])
named([  # a set's text is its hash table's order; `\N{...}` reads Unicode's names
    (P.joinedstr([fv(x)]), {"x": {1}}, "Python3 joinedstr's field of set is not simplified", "a"),
    (P.joinedstr([fv(x, "r")]), {"x": {1}}, "Python3 joinedstr's field of set is not simplified", "a"),
    (P.joinedstr(["\\N{BULLET}"]), {}, "Python3 joinedstr's \\N escape is not simplified", "a"),
])

<a id="PYS-05"></a>
## PYS-05 · `format()`'s mini-language for a str, an int, a bool, a float and a complex number, as CPython rounds and composes them; a locale's spec stays, named

In [ ]:
folds([(P.joinedstr([fv(x, None, (spec,))]), {"x": bound}, expected) for bound, spec, expected in [
    (1234, "08,", "0,001,234"), (-1, "06", "-00001"), (1, "=+6", "+    1"), (255, "#010_x", "0x000_00ff"), (65, ">3c", "  A"),
    (12345678, "_b", "1011_1100_0110_0001_0100_1110"), (True, ">5", "    1"), (255, "X", "FF"), (5, "+", "+5"), (5, " ", " 5"),
    (2**60, ".3e", "1.153e+18"), (12345.678, ",.2f", "12,345.68"), (2.675, ".2f", "2.67"), (-0.0, "z.1f", "0.0"),
    (1.5, "#.0f", "2."), (2.5, ".0f", "2"), (1234.5, ".3", "1.23e+03"), (1.0, ".3", "1.0"), (0.5, "%", "50.000000%"),
    (123456789.0, "g", "1.23457e+08"), (100.0, "#g", "100.000"), (1e-5, ".3e", "1.000e-05"), (1e-5, "E", "1.000000E-05"),
    (0.0, "e", "0.000000e+00"), (1e23, ".3e", "1.000e+23"), (9.9999, ".2e", "1.00e+01"), (1.5, "e", "1.500000e+00"),
    (1.5, ">6", "   1.5"), (1.5, " .1f", " 1.5"), (1e16, "", "1e+16"), (-inf, "E", "-INF"), (nan, "f", "nan"),
    (inf, "+f", "+inf"), (nan, "+.1f", "+nan"), ("ab", "^6", "  ab  "), ("ab", "0>5", "000ab"), ("héllo", ".2", "hé"),
    ("a", "*<3", "a**")]])
# A complex number: each part a float's, the imaginary part signed, then `j`; with no type, a repr's digits (or `g`'s,
# with a precision), in parentheses unless the real part is +0, which is left out; then padded as a whole
folds([(P.joinedstr([fv(x, None, (spec,))]), {"x": bound}, expected) for bound, spec, expected in [
    (1 + 2j, ">12", "      (1+2j)"), (1.23456 + 1j, ".3", "(1.23+1j)"), (2j, "+", "+2j"), (1 + 1j, "#", "(1.+1.j)"),
    (1e16 + 1j, "#", "(1.e+16+1.j)"), (1234.5 + 1j, "_.2f", "1_234.50+1.00j"), (1 + 2j, "E", "1.000000E+00+2.000000E+00j"),
    (complex(inf, nan), "F", "INF+NANj"), (complex(-0.0, -0.0), "z.1f", "0.0+0.0j"), (1 + 2j, "-^15", "----(1+2j)-----"),
    (1 + 2j, "*<9", "(1+2j)***"), (1 + 2j, " ", "( 1+2j)"), (-1.5 - 2.5j, "g", "-1.5-2.5j")]])
named([  # a locale's; a spec Python refuses (zero padding for a complex number); a spec of a value with no format of its own
    (P.joinedstr([fv(x, None, ("n",))]), {"x": 1}, "Python3 joinedstr's format spec 'n' of int is not simplified", "a"),
    (P.joinedstr([fv(c1, None, ("n",))]), {}, "Python3 joinedstr's format spec 'n' of complex is not simplified", "a"),
    (P.joinedstr([fv(c1, None, ("05",))]), {}, "Python3 joinedstr's format spec '05' of complex is not simplified", "a"),
    (P.joinedstr([fv(x, None, (">5",))]), {"x": {1}}, "Python3 joinedstr's format of set is not simplified", "a"),
])
fails([
    (P.joinedstr([fv(P.constant(None), None, (">5",))]), {}, TypeError, "unsupported format string passed to NoneType.__format__"),
    (P.joinedstr([fv(x, None, ("c",))]), {"x": 2**70}, OverflowError, "Python int too large to convert to C long"),
    (P.joinedstr([fv(x, None, ("c",))]), {"x": -1}, OverflowError, "%c arg not in range(0x110000)"),
])

<a id="PYS-06"></a>
## PYS-06 · t-strings: a template, as Python 3.14's `string.templatelib` holds it, with no literal; its residual kept as written

In [ ]:
template = P.templatestr(["a", P.interpolation(x, None, None, True, "x = "), "b", P.interpolation(y, "r", (">3",))])
folds([
    (P.subscript(P.attribute(template, "strings"), 0), {"x": 1, "y": "s"}, "ax = "),
    (P.attribute(P.subscript(P.attribute(template, "interpolations"), 0), "expression"), {"x": 1, "y": "s"}, "x"),
    (P.attribute(P.subscript(P.attribute(template, "interpolations"), 0), "conversion"), {"x": 1, "y": "s"}, "r"),
    (P.attribute(P.subscript(P.attribute(template, "interpolations"), 1), "format_spec"), {"x": 1, "y": "s"}, ">3"),
    (P.subscript(P.attribute(template, "values"), 1), {"x": 1, "y": "s"}, "s"),
    (P.subscript(P.attribute(P.templatestr([]), "strings"), 0), {}, ""),
    (P.subscript(P.attribute(P.templatestr(["a"], None), "strings"), 0), {}, "a"),
    (P.subscript(P.attribute(P.templatestr([P.interpolation(x, None, None, True)]), "strings"), 0), {"x": 1}, ""),
])
stays([(template, {}, "t'a{x = }b{y!r:>3}'"), (template, {"x": 1, "y": 2}, "t'a{x = }b{y!r:>3}'")])  # its text is its value's

<a id="PYS-07"></a>
## PYS-07 · Displays and unpacking: tuples, sets by Python's equality, dicts and their `**` items, `*` in a display or a call, and their operators

In [ ]:
folds([
    (P.compare("==", P.tuple_(1, st(v)), P.tuple_(1, 2, 3)), {"v": [2, 3]}, True), (P.compare("==", P.tuple_(1), P.list_(1)), {}, False),
    (P.call("len", P.set_(1, 1.0, True, 2)), {}, 2), (P.compare("in", 3, P.set_(st(v))), {"v": [3]}, True),
    (P.compare("<", P.set_(1), P.set_(1, 2)), {}, True), (P.call("len", P.binop("^", P.set_(1, 2), P.set_(2, 3))), {}, 2),
    (P.subscript(P.dict_((None, m), ("b", 2)), "a"), {"m": {"a": 1}}, 1), (P.call("max", st(v)), {"v": [1, 3, 2]}, 3),
    (P.compare("<", P.tuple_(1, 2), P.tuple_(1, 2, 0)), {}, True), (P.subscript(P.binop("+", P.tuple_(1), P.tuple_(2)), -1), {}, 2),
    (s(P.set_(1)), {}, "{1}"), (s(P.call("set")), {}, "set()"), (P.unaryop("not", P.tuple_()), {}, True),
    (P.compare("==", P.set_(1, 2), P.set_(2, 1)), {}, True), (P.compare("==", P.dict_(("a", 1)), m), {"m": {"a": 1}}, True),
    (P.compare("==", P.dict_(("a", 1)), P.dict_(("b", 1))), {}, False), (P.compare("<", P.list_(1, 2), P.list_(1, 3)), {}, True),
    (P.compare("<=", P.set_(1), P.set_(1)), {}, True), (P.compare(">", P.set_(1, 2), P.set_(1)), {}, True),
    (P.compare(">=", P.set_(1), P.set_(1, 2)), {}, False), (P.call("len", P.binop("*", P.tuple_(1, 2), 2)), {}, 4),
    (P.call("len", P.binop("+", P.list_(1), P.list_(2))), {}, 2), (P.call("len", P.binop("*", P.list_(1), 2)), {}, 2),
    (P.subscript(P.dict_((P.tuple_(1, 2), 3)), P.tuple_(1, 2)), {}, 3), (P.call("len", P.binop("&", P.set_(1, 2), P.set_(2, 3))), {}, 1),
    (P.call("len", P.binop("|", P.set_(1), P.set_(2))), {}, 2), (P.subscript(P.binop("|", P.dict_(("a", 1)), P.dict_(("b", 2))), "b"), {}, 2),
    (P.call("len", P.binop("-", P.set_(1, 2), P.set_(2))), {}, 1),
])
stays([
    (P.set_(P.binop("+", 1, 1), y), {}, "{2, y}"), (P.list_(st(v), 3), {"v": (1, 2)}, "[1, 2, 3]"), (P.list_(st(v)), {}, "[*v]"),
    (P.dict_((None, m), ("b", P.binop("+", 1, 1))), {"m": {"a": 1}}, "{'a': 1, 'b': 2}"), (P.dict_((None, m), ("b", 2)), {}, "{**m, 'b': 2}"),
])
fails([
    (P.list_(st(v)), {"v": 5}, TypeError, "Value after * must be an iterable, not int"),
    (P.dict_((None, m)), {"m": 1}, TypeError, "'int' object is not a mapping"),
    (P.set_(v), {"v": [1]}, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"),
    (P.dict_((P.tuple_(1, P.list_()), 2)), {}, TypeError, "cannot use 'tuple' as a dict key (unhashable type: 'list')"),
    (P.binop("+", P.list_(1), P.tuple_(2)), {}, TypeError, 'can only concatenate list (not "tuple") to list'),
    (P.list_(st(P.generatorexp(P.binop("/", 1, n("p")), comp("p", v)))), {"v": [0]}, ZeroDivisionError, "division by zero"),
])

<a id="PYS-08"></a>
## PYS-08 · Slices and subscripts by one: negative indices, steps and bounds clamped, as Python slices a sequence

In [ ]:
folds([
    (P.subscript("abcde", P.slice_(1, -1)), {}, "bcd"), (P.subscript("abcde", P.slice_(None, None, -2)), {}, "eca"),
    (P.subscript(b"abc", P.slice_(None, None, 2)), {}, b"ac"), (P.call("len", P.subscript(v, P.slice_(5, 1, -2))), {"v": [1] * 6}, 2),
    (P.attribute(P.slice_(1, 2), "step"), {}, None), (P.call("len", P.subscript(P.list_(1, 2, 3), P.slice_(-10, None, -1))), {}, 0),
    (P.subscript(P.subscript(P.list_(1, 2, 3), P.slice_(10, None, -1)), 0), {}, 3), (P.call("len", P.subscript(P.list_(1, 2, 3), P.slice_(-10))), {}, 3),
    (P.subscript(P.subscript(P.tuple_(1, 2, 3), P.slice_(1)), 0), {}, 2), (P.call("len", P.subscript(P.list_(1, 2, 3), P.slice_(10))), {}, 0),
])
stays([(P.subscript(v, P.slice_(1)), {"v": [1, 2, 3]}, "[2, 3]"), (P.subscript(v, P.slice_(P.binop("+", 0, 1))), {}, "v[1:]"),
       (P.slice_(1, 2), {}, "1:2")])  # a slice has no literal
fails([
    (P.subscript(v, P.slice_(None, None, 0)), {"v": [1]}, ValueError, "slice step cannot be zero"),
    (P.subscript(v, P.slice_(1.5)), {"v": [1]}, TypeError, "slice indices must be integers or None or have an __index__ method"),
    (P.subscript(P.tuple_(1, 2), 5), {}, IndexError, "tuple index out of range"),
])

<a id="PYS-09"></a>
## PYS-09 · Comprehensions: a list's, a set's and a dict's, of several clauses, over targets that unpack; a target that writes into data stays, named

In [ ]:
folds([
    (P.call("sum", P.listcomp(P.binop("*", n("p"), n("q")), comp(P.tuple_(n("p"), n("q")), v))), {"v": [(1, 2), (3, 4)]}, 14),
    (P.call("len", P.setcomp(P.binop("%", n("p"), 2), comp("p", v))), {"v": [1, 2, 3]}, 2),
    (P.subscript(P.dictcomp(n("p"), P.binop("*", n("p"), n("p")), comp("p", v, (P.compare(">", n("p"), 1),))), 3), {"v": [1, 2, 3]}, 9),
    (P.call("sum", P.generatorexp(P.call("len", n("q")), comp(P.tuple_(n("p"), st(n("q"))), v))), {"v": [[1, 2, 3], [4]]}, 2),
    (P.call("sum", P.listcomp(n("p"), comp(P.parenthesized(n("p")), v))), {"v": [1, 2]}, 3),
    (P.call("sum", P.listcomp(n("q"), comp("p", v), comp("q", n("p")))), {"v": [[1, 2], [3]]}, 6),
    (P.call("sum", P.generatorexp(st(n("p")), comp("p", v))), {"v": [[1, 2], [3]]}, 6),  # `(*p for p in v)`, Python 3.15
    (P.subscript(P.dictcomp(m, None, comp("m", v)), "b"), {"v": [{"a": 1}, {"b": 2}]}, 2),  # `{**m for m in v}`, Python 3.15
])
stays([
    (P.listcomp(P.binop("*", n("p"), 2), comp("p", v)), {"v": [1, 2]}, "[2, 4]"),
    (P.listcomp(P.binop("*", n("p"), P.binop("+", 1, 1)), comp("p", v)), {}, "[p * 2 for p in v]"),
    (P.dictcomp(n("p"), 0, comp("p", "ab")), {}, "{'a': 0, 'b': 0}"), (P.setcomp(n("p"), comp("p", v)), {"v": [1]}, "{1}"),
])
named([  # a target that writes into data, and an `async for`, need a statement (d); two starred targets are no syntax (b)
    (P.listcomp(n("p"), comp(P.tuple_(n("p"), P.subscript(n("o"), 0)), v)), {"v": [1]}, "Python3 subscript target is not simplified", "d"),
    (P.listcomp(n("p"), comp(P.attribute(n("o"), "a"), v)), {"v": [1]}, "Python3 attribute target is not simplified", "d"),
    (P.listcomp(n("p"), comp("p", v, (), True)), {"v": [1]}, "Python3 async comprehension is not simplified", "d"),
    (P.listcomp(n("p"), comp(P.tuple_(st(n("p")), st(n("q"))), v)), {"v": [1]}, "Python3 starred target is not simplified", "b"),
])
fails([
    (P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), {"v": [1]}, TypeError, "cannot unpack non-iterable int object"),
    (P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), {"v": [[1, 2, 3]]}, ValueError, "too many values to unpack (expected 2, got 3)"),
    (P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), {"v": [[1]]}, ValueError, "not enough values to unpack (expected 2, got 1)"),
    (P.listcomp(n("p"), comp(P.tuple_(n("p"), st(n("q")), n("r")), v)), {"v": [[1]]}, ValueError,
     "not enough values to unpack (expected at least 2, got 1)"),
])

<a id="PYS-10"></a>
## PYS-10 · Lambdas and keywords: closures binding Python's parameters, with CPython's messages, and the builtins' keywords

In [ ]:
folds([
    (P.call(P.lambda_(P.binop("+", n("a"), n("b")), ["a", P.arg("b", 10)]), 1), {}, 11),
    (P.call(P.lambda_(n("k"), kwonlyargs=["k"]), kw("k", 4)), {}, 4), (P.call(P.lambda_(P.call("len", n("r")), vararg="r"), 1, 2, 3), {}, 3),
    (P.call(P.lambda_(P.subscript(n("o"), "z"), kwarg="o"), kw(None, m)), {"m": {"z": 5}}, 5),
    (P.call(P.lambda_(n("a"), posonlyargs=["a"]), 7), {}, 7), (P.call(P.lambda_(n("a"), ["a"]), st(v)), {"v": [8]}, 8),
    (P.call("max", v, kw("key", P.lambda_(P.unaryop("-", x), ["x"]))), {"v": [3, 1, 2]}, 1),
    (P.call("min", v, kw("default", 0)), {"v": []}, 0), (P.call("sum", v, kw("start", 10)), {"v": [1, 2]}, 13),
])
# A closure has no literal: a lambda stays, its body simplified in the scope it is made in
stays([
    (P.lambda_(P.binop("+", x, P.binop("+", 1, 2)), ["x"]), {}, "lambda x: x + 3"),
    (P.lambda_(P.binop("+", x, y), ["x"]), {"y": 2}, "lambda x: x + 2"), (P.call(P.lambda_(x, ["x"]), z), {}, "(lambda x: x)(z)"),
])
fails([
    (P.call(P.lambda_(n("a"), ["a"])), {}, TypeError, "<lambda>() missing 1 required positional argument: 'a'"),
    (P.call(P.lambda_(n("a"), ["a", "b", "c"])), {}, TypeError, "<lambda>() missing 3 required positional arguments: 'a', 'b', and 'c'"),
    (P.call(P.lambda_(n("a"), ["a", "b"]), 1, 2, 3), {}, TypeError, "<lambda>() takes 2 positional arguments but 3 were given"),
    (P.call(P.lambda_(n("a"), ["a", P.arg("b", 1)]), 1, 2, 3), {}, TypeError, "<lambda>() takes from 1 to 2 positional arguments but 3 were given"),
    (P.call(P.lambda_(0), 1), {}, TypeError, "<lambda>() takes 0 positional arguments but 1 was given"),
    (P.call(P.lambda_(n("a"), ["a"], kwonlyargs=["k"]), 1, 2, kw("k", 3)), {}, TypeError,
     "<lambda>() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given"),
    (P.call(P.lambda_(0, kwonlyargs=["k", "j"]), 1, kw("k", 1), kw("j", 2)), {}, TypeError,
     "<lambda>() takes 0 positional arguments but 1 positional argument (and 2 keyword-only arguments) were given"),
    (P.call(P.lambda_(n("a"), ["a"]), 1, kw("a", 2)), {}, TypeError, "<lambda>() got multiple values for argument 'a'"),
    (P.call(P.lambda_(n("a"), ["a"]), kw("b", 2)), {}, TypeError, "<lambda>() got an unexpected keyword argument 'b'"),
    (P.call(P.lambda_(n("k"), kwonlyargs=["k"])), {}, TypeError, "<lambda>() missing 1 required keyword-only argument: 'k'"),
    (P.call(P.lambda_(n("a"), posonlyargs=["a"]), kw("a", 1)), {}, TypeError,
     "<lambda>() got some positional-only arguments passed as keyword arguments: 'a'"),
    (P.call(P.lambda_(n("a"), ["a"]), kw("a", 1), kw(None, m)), {"m": {"a": 2}}, TypeError, "<lambda>() got multiple values for keyword argument 'a'"),
    (P.call(P.lambda_(n("a"), ["a"]), kw(None, m)), {"m": {1: 5}}, TypeError, "keywords must be strings"),
    (P.call(P.lambda_(n("a"), ["a"]), kw(None, m)), {"m": [1]}, TypeError, "<lambda>() argument after ** must be a mapping, not list"),
    (P.call(P.lambda_(n("a"), ["a"]), st(v)), {"v": 5}, TypeError, "<lambda>() argument after * must be an iterable, not int"),
    (P.call("max", v, kw("foo", 1)), {"v": [1]}, TypeError, "max() got an unexpected keyword argument 'foo'"),
    (P.call("min", 1, 2, kw("default", 0)), {}, TypeError, "Cannot specify a default for min() with multiple positional arguments"),
    (P.call("abs", kw("x", 1)), {}, TypeError, "abs() takes no keyword arguments"),
])
# A function the scope provides takes keywords as Python's do (in TypeScript, none: a JavaScript function has no
# keyword parameters)
assert value(P.call(n("f"), kw("k", 1)), {"f": lambda **k: k["k"]}) == 1

<a id="PYS-11"></a>
## PYS-11 · `is` and `is not` by identity, and `@`: whether equal immutable values are one object Python leaves to its implementation, which stays, named

In [ ]:
folds([
    (P.compare("is", x, P.constant(None)), {"x": None}, True), (P.compare("is not", 1, P.constant(None)), {}, True),
    (P.compare("is", x, x), {"x": [1]}, True), (P.compare("is", P.list_(1), P.list_(1)), {}, False),
    (P.compare("is", 1, 1.0), {}, False), (P.compare("is", True, True), {}, True), (P.compare("is", 1, 2), {}, False),
    (P.compare("is", P.constant(...), P.constant(...)), {}, True), (P.compare("is not", x, y), {"x": "a", "y": "b"}, True),
])
named([
    (P.compare("is", 1, 1), {}, "Python3 compare 'is' of equal immutable values is not simplified", "b"),
    (P.compare("is", x, y), {"x": nan, "y": nan}, "Python3 compare 'is' of equal immutable values is not simplified", "b"),
    (P.compare("is not", x, y), {"x": "a", "y": "a"}, "Python3 compare 'is' of equal immutable values is not simplified", "b"),
])
# `@` calls an operand's `__matmul__`, which no builtin type has
fails([(P.binop("@", x, y), {"x": [1], "y": [2]}, TypeError, "unsupported operand type(s) for @: 'list' and 'list'")])

<a id="PYS-12"></a>
## PYS-12 · What stays, by its class: `yield`, `yield from` and `await` (d), invalid syntax and a kind of the family's other dialects (b); `FOLDED`

In [ ]:
named([
    (P.yield_(1), {}, "Python3 yield is not simplified", "d"), (P.yieldfrom(v), {"v": [1]}, "Python3 yieldfrom is not simplified", "d"),
    (P.await_(x), {"x": 1}, "Python3 await is not simplified", "d"),
    (lambda u: u.missing(lambda r: r), {}, "Python3 missing is not simplified", "b"),
])
# What stays keeps its subterms simplified, and what does not depend on it folds around it
assert text(P.binop("+", P.await_(P.binop("+", 1, 2)), 1)) == "await 3 + 1"
assert reasons(P.binop("+", P.await_(P.binop("+", 1, 2)), 1)) == [("Python3 await is not simplified", "d")]
assert text(P.boolop("or", True, P.yield_(1))) == "True"
# A kind of Python Types, Python Patterns or Python RegExp is its own dialect's, of class (b): simplification across a
# sub-dialect (mbse-basis CAP-299) is to reach it
assert [PV.SIMPLIFIER.category(dialect, "typevar", "typevar") for dialect in ("Python Types", "Python Patterns", "Python RegExp")] == ["b"] * 3
assert PV.SIMPLIFIER.category("Python3", "binop '**' of a complex power", "binop") == "a"
assert PV.FOLDED == {"literal", "constant", "variable", "let", "list", "tuple", "set", "dict", "starred", "parenthesized",
                     "concatenatedstring", "joinedstr", "templatestr", "attribute", "subscript", "slice", "call", "compare",
                     "boolop", "binop", "unaryop", "ifexp", "namedexpr", "lambda", "listcomp", "setcomp", "dictcomp",
                     "generatorexp", "import", "importfrom"}
# What Python may never evaluate stays, though it would raise: a branch, by an unknown test; what follows an unknown
# operand of `and` or `or`, or a chain's first comparison; a lambda's body
div = P.binop("/", 1, 0)
for spec, written in [(P.ifexp(x, div, 2), "1 / 0 if x else 2"), (P.boolop("and", x, div), "x and 1 / 0"),
                      (P.compare("<", x, 1, ("<", div)), "x < 1 < 1 / 0"), (P.lambda_(div), "lambda: 1 / 0")]:
    assert text(spec) == written and reasons(spec) == [], text(spec)
with raises(ZeroDivisionError, match="division by zero"):  # taken, it raises
    text(P.ifexp(True, div, 2))

<a id="PYS-13"></a>
## PYS-13 · Sets: a non-empty set written as its display, its items in an order both implementations compute; an empty set stays

In [ ]:
# A non-empty set is written as its display, a closed constructor expression, its items in an order of their values
# that both implementations compute, not its hash table's: None, `...`, real numbers (a NaN after every other), complex
# numbers (by their real parts, then their imaginary ones), strs and bytes as Python orders them, then tuples, item by
# item
stays([
    (x, {"x": {10, 9, 2.5, True}}, "{True, 2.5, 9, 10}"), (x, {"x": {"b", "a", "é", "B"}}, "{'B', 'a', 'b', 'é'}"),
    (x, {"x": {None, ..., 1j, 1, "a", b"z", (1, 2), (1,), (0, "z")}}, "{None, ..., 1, 1j, 'a', b'z', (0, 'z'), (1,), (1, 2)}"),
    (x, {"x": {nan, -inf, 1}}, "{-float('inf'), 1, float('nan')}"), (x, {"x": {b"b", b"a"}}, "{b'a', b'b'}"),
    (x, {"x": {complex(nan, 1), 1 + 2j, 2j, 1j}}, "{1j, 2j, (1+2j), (nan+1j)}"),
    (x, {"x": {(None, 1), (None, 0), (..., 0)}}, "{(None, 0), (None, 1), (..., 0)}"),
    (P.set_(3, 1, P.binop("+", 1, 1)), {}, "{1, 2, 3}"), (P.setcomp(P.binop("%", n("p"), 3), comp("p", v)), {"v": [5, 4]}, "{1, 2}"),
    (x, {"x": [1, {2}]}, "[1, {2}]"), (x, {"x": {"k": {2, 1}}}, "{'k': {1, 2}}"),
])
# Its root a closed constructor expression, not a literal, the simplification evaluated the set
assert simplified(x, {"x": {2, 1}}).constructed and not simplified(x, {"x": {2, 1}}).literal
assert value(P.set_(2, 1)) == {1, 2} and type(value(P.set_(2, 1))) is set
# The framework's `known`, which a dialect's `simplify` hook calls with a value it computed, writes a set's display too
known = PV.SIMPLIFIER.known(term(x), {2, 1})
assert known.constructed and not known.literal and PText.ToText(term(known.expression)) == "{1, 2}"
# An empty set has no display (`{}` is a dict, and `set()` calls a name), nor has a set of an item with none: each stays
stays([(x, {"x": set()}, "x"), (P.call("set"), {}, "set()"), (x, {"x": [set()]}, "x"),
       (P.set_(P.lambda_(1)), {}, "{lambda: 1}")])
with raises(NotEvaluated, match="the root is not a closed constructor expression: its value has no closed constructor expression"):
    simplified(P.binop("-", P.set_(1), P.set_(1))).evaluate()  # its value an empty set